# Qwen3.8 Flash-Next FP8 × FreeToken-style A100 runtime

Target: **Google Colab A100 80 GiB + high-RAM runtime**, text-only. This notebook exercises the custom heterogeneous MoE runtime in `All-testing`.

Architecture: routed experts stay authoritative in CPU RAM as FP8; always-used components run on A100; decode uses a global `(layer, expert)` LRU; long prefill can use two full-layer FP8 GPU buffers; and the runtime records host→device bytes, cache hits/misses and throughput.

> The official FP8 checkpoint is very large. Run the hardware/disk preflight before starting the model download. This is a research reproduction path, not a claim that the paper's desktop throughput has already been reproduced on Colab.

In [ ]:
# 1. Install the exact research branch
!rm -rf /content/All-testing
!git clone --depth 1 --branch qwen38-flash-freetoken-colab https://github.com/Logan17de/All-testing.git /content/All-testing
%cd /content/All-testing/llm
%pip install -q -U -r qwen38_flash_freetoken/requirements-colab.txt
%pip install -q -U -e ".[qwen38-flash-freetoken]"
print('Install complete ✅')

In [ ]:
# 2. A100 / RAM / disk preflight — cheap and safe to run first
import json, subprocess
from qwen38_flash_freetoken.config import RuntimeConfig
from qwen38_flash_freetoken.hardware import collect_hardware_report, validate_colab_target
cfg = RuntimeConfig(cache_dir='/content/hf_cache', expert_cache_gib=42.0, cache_format='bf16', max_context_tokens=32768)
report = collect_hardware_report(cfg.cache_dir)
print(json.dumps(report.as_dict(), indent=2))
print('Problems:', validate_colab_target(cfg, strict=False))
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['free','-h'], check=False)
subprocess.run(['df','-h','/content'], check=False)
validate_colab_target(cfg, strict=True)
print('Target runtime looks usable ✅')

In [ ]:
# 3. Optional Hugging Face token from Colab Secrets
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        print('HF_TOKEN loaded ✅')
except Exception as exc:
    print('HF token not configured; public download will be used:', exc)
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE','1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY','1')

In [ ]:
# 4. Inspect the live checkpoint and fail on architecture drift before the huge download
from qwen38_flash_freetoken.manifest import inspect_remote_model, validate_manifest
from qwen38_flash_freetoken.planner import build_memory_plan
manifest = inspect_remote_model(cfg)
print(json.dumps(manifest.as_dict(), indent=2))
print(validate_manifest(manifest, cfg))
plan = build_memory_plan(cfg)
print('Memory plan:')
print(json.dumps(plan.as_dict(), indent=2))

In [ ]:
# 5. Measure this VM's real host/device bandwidth
from dataclasses import asdict
from qwen38_flash_freetoken.hardware import profile_bandwidth
bw = profile_bandwidth()
print(json.dumps(asdict(bw), indent=2))

## 6. Load the model
This is the expensive step. The loader keeps routed expert banks in host RAM, removes Accelerate execution hooks from those expert modules, converts CUDA-resident non-expert FP8 linears to BF16 for A100 compute, builds the global expert cache, and binds the custom `freetoken` Experts backend.

In [ ]:
from qwen38_flash_freetoken.loader import load_qwen_runtime
loaded = load_qwen_runtime(cfg, measure_bandwidth=True, strict_hardware=True)
print('MODEL LOADED ✅')
print('Bound expert layers:', len(loaded.runtime.expert_modules))
print('GPU expert slots:', loaded.runtime.cache.slots)
print('GPU expert-cache GiB:', loaded.runtime.cache.bytes_allocated / 1024**3)
print('Prefill double buffer:', loaded.runtime.prefill_buffers is not None)

In [ ]:
# 7. Residency sanity check
import torch
expert_devices = sorted({str(m.gate_up_proj.device) for m in loaded.runtime.expert_modules.values()})
print('Routed expert devices:', expert_devices)
assert expert_devices == ['cpu'], expert_devices
ple = []
for name, module in loaded.model.named_modules():
    if name.endswith('ple.ple_embedding.ngram_embedding') and hasattr(module, 'weight'):
        ple.append((name, str(module.weight.device), str(module.weight.dtype)))
print('PLE/n-gram modules:', ple)
assert ple and all(device == 'cpu' for _, device, _ in ple), ple
free, total = torch.cuda.mem_get_info()
print(f'A100: used={(total-free)/1024**3:.2f} GiB | free={free/1024**3:.2f} GiB | total={total/1024**3:.2f} GiB')
print('Residency checks passed ✅')

In [ ]:
# 8. Deterministic generation smoke test
from qwen38_flash_freetoken.inference import generate_text
result = generate_text(loaded, [{'role':'user','content':'Explain in one short sentence why an MoE expert cache helps inference.'}], max_new_tokens=48, temperature=0.0)
print(result.text)
print(f'prompt={result.prompt_tokens} completion={result.completion_tokens} elapsed={result.elapsed_s:.2f}s TPS={result.tokens_per_second:.2f}')
print(json.dumps(loaded.runtime.stats.as_dict(), indent=2))

In [ ]:
# 9. Cold/warm-cache comparison
prompt = [{'role':'user','content':'Explain how a CPU-RAM expert pool and a GPU LRU cache cooperate in MoE inference.'}]
rows = []
for run in range(3):
    before = loaded.runtime.stats.as_dict().copy()
    r = generate_text(loaded, prompt, max_new_tokens=64, temperature=0.0)
    after = loaded.runtime.stats.as_dict().copy()
    rows.append({'run':run+1,'tps':r.tokens_per_second,'seconds':r.elapsed_s,'completion_tokens':r.completion_tokens,'new_cache_hits':after['cache_hits']-before['cache_hits'],'new_cache_misses':after['cache_misses']-before['cache_misses'],'new_h2d_gib':(after['h2d_bytes']-before['h2d_bytes'])/1024**3})
print(json.dumps(rows, indent=2))
print('Cumulative metrics:')
print(json.dumps(loaded.runtime.stats.as_dict(), indent=2))

In [ ]:
# 10. Optional private localhost OpenAI-compatible API
import secrets, time, requests
from qwen38_flash_freetoken.serve import start_server_thread
api_key = secrets.token_urlsafe(24)
server_thread = start_server_thread(loaded, api_key, host='127.0.0.1', port=8000)
for _ in range(60):
    try:
        r = requests.get('http://127.0.0.1:8000/v1/models', headers={'Authorization':f'Bearer {api_key}'}, timeout=2)
        if r.ok: break
    except Exception: pass
    time.sleep(1)
print(r.json())

## 11. Optional Harness relay
Add `QWEN_RELAY_SECRET` to Colab Secrets. The helper reuses the repository's outbound-only Supabase relay, so no public Colab port is opened. If you already started the localhost API above on port 8000, restart the runtime and skip that API cell before using the blocking relay cell below.

In [ ]:
# Uncomment to serve Harness jobs with this already-loaded model:
# from qwen38_flash_freetoken.relay import serve_loaded_to_harness
# serve_loaded_to_harness(loaded, cfg)

## What to save from the first A100 run
Keep the outputs of the preflight, manifest, bandwidth profile, residency check, deterministic generation, and warm-cache comparison. Those measurements tell us whether the next optimization should be device-side LRU/CUDA Graphs, better pinned staging, or the real SIMD CPU expert kernel + `q*` co-execution.